In [ ]:
"""
Adaptive Depth Limiting Discovery Framework
============================================

Instead of using a fixed depth limit (e.g., max_depth=2) for 3 months,
systematically vary the depth limit across different phases to understand:

1. How does system behavior change with different depth limits?
2. At what depth do workflows naturally stabilize?
3. Where are the optimal trade-offs between autonomy and orchestration?
4. Which workflows benefit from deeper cascading vs. which need orchestration?

This produces richly layered training data showing:
- Performance curves across depth levels
- Optimal depth for each workflow
- Cascading benefits/diminishing returns
- Error patterns at different depths
- Resource utilization across depth variations

Phases:
------
Phase 1 (Days 1-7):   max_depth=1   - Agents cannot call other agents
Phase 2 (Days 8-21):  max_depth=2   - Single level of agent-to-agent calls
Phase 3 (Days 22-35): max_depth=3   - Two levels of cascading
Phase 4 (Days 36-49): max_depth=2   - Back to depth=2 (validation & comparison)
Phase 5 (Days 50-63): max_depth=4   - Three levels (test outer bounds)
Phase 6 (Days 64-75): max_depth=2   - Final validation (back to baseline)
Phase 7 (Days 76-90): adaptive      - AI-controlled depth per workflow (optional)

This A/B testing approach reveals:
- Which depth limits enable which workflows
- System performance degradation curves
- Latency, error rate, cost implications at each depth
- Optimal depth for production orchestrator
"""

import json
import statistics
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict, Counter
from enum import Enum


# ============================================================================
# PART 1: ADAPTIVE DEPTH EXPERIMENT FRAMEWORK
# ============================================================================

class DepthPhase(str, Enum):
    """Discovery phases with different depth limits"""
    PHASE_1_DEPTH_1 = "phase_1_depth_1"      # Days 1-7
    PHASE_2_DEPTH_2 = "phase_2_depth_2"      # Days 8-21
    PHASE_3_DEPTH_3 = "phase_3_depth_3"      # Days 22-35
    PHASE_4_DEPTH_2_VALIDATION = "phase_4_depth_2_validation"  # Days 36-49
    PHASE_5_DEPTH_4 = "phase_5_depth_4"      # Days 50-63
    PHASE_6_DEPTH_2_FINAL = "phase_6_depth_2_final"            # Days 64-75
    PHASE_7_ADAPTIVE = "phase_7_adaptive"    # Days 76-90


@dataclass
class DepthExperimentConfig:
    """Configuration for a depth limit experiment phase"""
    phase_id: str
    max_depth: int
    start_date: datetime
    end_date: datetime
    duration_days: int
    description: str
    hypothesis: str
    expected_outcomes: List[str]
    
    def to_dict(self) -> Dict:
        return {
            "phase_id": self.phase_id,
            "max_depth": self.max_depth,
            "start_date": self.start_date.isoformat(),
            "end_date": self.end_date.isoformat(),
            "duration_days": self.duration_days,
            "description": self.description,
            "hypothesis": self.hypothesis,
            "expected_outcomes": self.expected_outcomes
        }


@dataclass
class WorkflowPerformanceByDepth:
    """How a workflow performs at a specific depth limit"""
    workflow_id: str
    depth_limit: int
    phase_id: str
    
    # Frequency metrics
    attempted: int = 0
    succeeded: int = 0
    failed: int = 0
    blocked_at_depth_limit: int = 0
    
    # Performance metrics
    avg_latency_ms: float = 0.0
    p95_latency_ms: float = 0.0
    p99_latency_ms: float = 0.0
    success_rate: float = 0.0
    
    # Cascading metrics
    max_depth_reached: int = 0
    avg_depth_used: float = 0.0
    
    # Cost metrics (simplified: latency as proxy for compute cost)
    total_cost_units: float = 0.0
    
    # Error breakdown
    timeout_errors: int = 0
    other_errors: int = 0
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class DepthPhaseAnalysis:
    """Analysis of a complete discovery phase"""
    phase_id: str
    max_depth: int
    start_date: str
    end_date: str
    
    total_calls: int = 0
    total_succeeded: int = 0
    total_failed: int = 0
    total_blocked: int = 0
    
    system_success_rate: float = 0.0
    system_avg_latency_ms: float = 0.0
    system_p95_latency_ms: float = 0.0
    
    workflow_performance: Dict[str, Dict] = field(default_factory=dict)
    agent_performance: Dict[str, Dict] = field(default_factory=dict)
    
    enabled_workflows: List[str] = field(default_factory=list)
    blocked_workflows: List[str] = field(default_factory=list)
    
    insights: List[str] = field(default_factory=list)
    
    def to_dict(self) -> Dict:
        return asdict(self)


# ============================================================================
# PART 2: ADAPTIVE DEPTH DISCOVERY ANALYZER
# ============================================================================

class AdaptiveDepthAnalyzer:
    """
    Analyze A2A logs across multiple depth limit phases.
    
    Compares how system behavior changes when depth limit is varied.
    Produces comprehensive training data showing optimal depth for
    different workflow patterns.
    
    CONTROL PHASE STRATEGY:
    =====================
    Phases 2, 4, and 6 use depth=2 intentionally as controls.
    This allows detection of:
    - System drift over time (agents learning, caching improving)
    - External factors (load patterns, time effects)
    - Validity of depth=3 and depth=4 measurements
    
    If Phase 2, 4, 6 metrics are consistent → depth differences are real
    If Phase 2, 4, 6 metrics diverge → system is changing → account for drift
    """
    
    def __init__(self, call_logs: List[Dict]):
        """
        Initialize with complete call logs from all phases.
        
        Call logs must include 'phase_id' field indicating which
        depth limiting phase the call occurred in.
        
        Args:
            call_logs: List of A2A call log entries with additional 'phase_id'
        """
        self.logs = call_logs
        self.phases_data: Dict[str, List[Dict]] = defaultdict(list)
        self.phase_analyses: Dict[str, DepthPhaseAnalysis] = {}
        self.control_phases = [
            "phase_2_depth_2",
            "phase_4_depth_2_validation", 
            "phase_6_depth_2_final"
        ]
        
        # Organize logs by phase
        for log in call_logs:
            phase_id = log.get('phase_id', 'unknown')
            self.phases_data[phase_id].append(log)
        
        # Analyze each phase
        self._analyze_all_phases()
    
    def _analyze_all_phases(self) -> None:
        """Analyze each depth limit phase"""
        for phase_id, logs in self.phases_data.items():
            analysis = self._analyze_phase(phase_id, logs)
            self.phase_analyses[phase_id] = analysis
    
    def _analyze_phase(self, phase_id: str, logs: List[Dict]) -> DepthPhaseAnalysis:
        """Analyze a single phase"""
        
        # Extract phase metadata
        first_log = logs[0] if logs else {}
        max_depth = first_log.get('max_depth', 2)
        start_date = first_log.get('timestamp', '')
        end_date = logs[-1].get('timestamp', '') if logs else ''
        
        analysis = DepthPhaseAnalysis(
            phase_id=phase_id,
            max_depth=max_depth,
            start_date=start_date,
            end_date=end_date,
            total_calls=len(logs)
        )
        
        # Aggregate statistics
        latencies = []
        workflow_stats: Dict[str, Dict] = defaultdict(lambda: {
            'attempted': 0, 'succeeded': 0, 'failed': 0, 'blocked': 0,
            'latencies': []
        })
        
        agent_stats: Dict[str, Dict] = defaultdict(lambda: {
            'calls': 0, 'success': 0, 'latencies': []
        })
        
        for log in logs:
            status = log.get('status', 'unknown')
            workflow = f"{log.get('caller')} → {log.get('target')}"
            latency = log.get('latency_ms', 0)
            
            # Overall stats
            if status == 'success':
                analysis.total_succeeded += 1
            elif status == 'depth_limit_exceeded':
                analysis.total_blocked += 1
                analysis.blocked_workflows.append(workflow)
            else:
                analysis.total_failed += 1
            
            # Workflow stats
            workflow_stats[workflow]['attempted'] += 1
            if status == 'success':
                workflow_stats[workflow]['succeeded'] += 1
                latencies.append(latency)
                workflow_stats[workflow]['latencies'].append(latency)
            elif status == 'depth_limit_exceeded':
                workflow_stats[workflow]['blocked'] += 1
            else:
                workflow_stats[workflow]['failed'] += 1
            
            # Agent stats
            target = log.get('target')
            if target:
                agent_stats[target]['calls'] += 1
                if status == 'success':
                    agent_stats[target]['success'] += 1
                    agent_stats[target]['latencies'].append(latency)
        
        # Calculate percentiles
        if latencies:
            analysis.system_success_rate = analysis.total_succeeded / len(logs)
            analysis.system_avg_latency_ms = statistics.mean(latencies)
            analysis.system_p95_latency_ms = sorted(latencies)[
                int(len(latencies) * 0.95)
            ]
        
        # Store workflow details
        for workflow, stats in workflow_stats.items():
            if stats['attempted'] > 0:
                success_rate = stats['succeeded'] / stats['attempted']
                analysis.workflow_performance[workflow] = {
                    'attempted': stats['attempted'],
                    'succeeded': stats['succeeded'],
                    'blocked': stats['blocked'],
                    'success_rate': success_rate,
                    'avg_latency_ms': statistics.mean(stats['latencies']) 
                                     if stats['latencies'] else 0
                }
                
                if success_rate > 0.8:
                    if workflow not in analysis.enabled_workflows:
                        analysis.enabled_workflows.append(workflow)
        
        # Store agent details
        for agent, stats in agent_stats.items():
            if stats['calls'] > 0:
                success_rate = stats['success'] / stats['calls']
                analysis.agent_performance[agent] = {
                    'calls': stats['calls'],
                    'success_rate': success_rate,
                    'avg_latency_ms': statistics.mean(stats['latencies']) 
                                     if stats['latencies'] else 0
                }
        
        # Generate insights
        analysis.insights = self._generate_phase_insights(
            phase_id, max_depth, analysis, len(logs)
        )
        
        return analysis
    
    def _generate_phase_insights(
        self, 
        phase_id: str, 
        max_depth: int, 
        analysis: DepthPhaseAnalysis,
        total_calls: int
    ) -> List[str]:
        """Generate insights for a phase"""
        
        insights = []
        
        # Insight 1: What workflows are enabled?
        if analysis.enabled_workflows:
            insights.append(
                f"Depth limit {max_depth} enables {len(analysis.enabled_workflows)} workflows: "
                f"{', '.join(analysis.enabled_workflows[:3])}"
            )
        
        # Insight 2: Are workflows blocked?
        if analysis.blocked_workflows:
            insights.append(
                f"With depth={max_depth}, {len(analysis.blocked_workflows)} workflows "
                f"hit depth limits ({analysis.total_blocked} times total)"
            )
        
        # Insight 3: Performance characteristics
        insights.append(
            f"System success rate: {analysis.system_success_rate*100:.1f}% | "
            f"Avg latency: {analysis.system_avg_latency_ms:.0f}ms | "
            f"P95: {analysis.system_p95_latency_ms:.0f}ms"
        )
        
        # Insight 4: Complexity assessment
        avg_depth_needed = 1
        for log in self.logs:
            if log.get('phase_id') == phase_id and log.get('status') == 'success':
                avg_depth_needed = max(avg_depth_needed, log.get('depth', 0) + 1)
        
        if avg_depth_needed > max_depth:
            insights.append(
                f"Average workflows need depth~{avg_depth_needed}, but limit is {max_depth}"
            )
        else:
            insights.append(
                f"Depth limit {max_depth} exceeds average need (~{avg_depth_needed})"
            )
        
        return insights
    
    def compare_phases(self) -> Dict:
        """
        Compare performance across all depth phases.
        
        Returns:
            Comparison showing how each metric changes with depth
        """
        
        phases_sorted = sorted(
            self.phase_analyses.items(),
            key=lambda x: x[1].max_depth
        )
        
        comparison = {
            "phases_ordered_by_depth": [],
            "performance_curves": {
                "success_rate": [],
                "avg_latency": [],
                "enabled_workflows": [],
                "blocked_workflows": []
            },
            "optimal_depth_by_metric": {},
            "diminishing_returns": []
        }
        
        # Build comparison data
        for phase_id, analysis in phases_sorted:
            depth = analysis.max_depth
            
            comparison["phases_ordered_by_depth"].append({
                "phase_id": phase_id,
                "depth": depth,
                "success_rate": analysis.system_success_rate,
                "avg_latency_ms": analysis.system_avg_latency_ms,
                "enabled_workflows": len(analysis.enabled_workflows),
                "blocked_workflows": len(analysis.blocked_workflows)
            })
            
            comparison["performance_curves"]["success_rate"].append({
                "depth": depth,
                "rate": analysis.system_success_rate
            })
            
            comparison["performance_curves"]["avg_latency"].append({
                "depth": depth,
                "ms": analysis.system_avg_latency_ms
            })
            
            comparison["performance_curves"]["enabled_workflows"].append({
                "depth": depth,
                "count": len(analysis.enabled_workflows)
            })
        
        # Find optimal depth for each metric
        # Optimal success rate (highest)
        best_success = max(
            comparison["performance_curves"]["success_rate"],
            key=lambda x: x["rate"]
        )
        comparison["optimal_depth_by_metric"]["success_rate"] = best_success["depth"]
        
        # Optimal latency (lowest, but above minimum depth)
        sorted_by_latency = sorted(
            comparison["performance_curves"]["avg_latency"],
            key=lambda x: x["ms"]
        )
        if sorted_by_latency:
            # Find sweet spot: lowest latency with reasonable workflow enablement
            best_latency = sorted_by_latency[0]
            comparison["optimal_depth_by_metric"]["latency"] = best_latency["depth"]
        
        # Detect diminishing returns
        prev_improvement = float('inf')
        for i in range(1, len(comparison["performance_curves"]["enabled_workflows"])):
            current = comparison["performance_curves"]["enabled_workflows"][i]
            prev = comparison["performance_curves"]["enabled_workflows"][i-1]
            
            improvement = current["count"] - prev["count"]
            
            if improvement < prev_improvement * 0.5:  # >50% reduction in improvement
                comparison["diminishing_returns"].append({
                    "between_depth": f"{prev['depth']} and {current['depth']}",
                    "workflow_gain": improvement,
                    "note": "Increasing depth no longer yields proportional benefits"
                })
            
            prev_improvement = improvement
        
        return comparison
    
    def workflow_optimal_depth(self) -> Dict:
        """
        For each workflow, determine optimal depth.
        
        Shows which workflows benefit from deeper cascading and which
        should be handled by orchestration instead.
        
        Returns:
            {
                "workflow_id": {
                    "optimal_depth": int,
                    "success_at_depth": {depth: success_rate},
                    "latency_at_depth": {depth: latency_ms},
                    "recommendation": "orchestrate|allow_autonomy|shallow_cascade"
                }
            }
        """
        
        workflow_depth_analysis: Dict[str, Dict] = defaultdict(
            lambda: {
                "success_at_depth": {},
                "latency_at_depth": {},
                "frequency_at_depth": {}
            }
        )
        
        # Collect metrics for each workflow at each depth
        for phase_id, analysis in self.phase_analyses.items():
            depth = analysis.max_depth
            
            for workflow, perf in analysis.workflow_performance.items():
                workflow_depth_analysis[workflow]["success_at_depth"][depth] = \
                    perf.get('success_rate', 0)
                workflow_depth_analysis[workflow]["latency_at_depth"][depth] = \
                    perf.get('avg_latency_ms', 0)
                workflow_depth_analysis[workflow]["frequency_at_depth"][depth] = \
                    perf.get('attempted', 0)
        
        # Analyze each workflow
        result = {}
        for workflow, data in workflow_depth_analysis.items():
            if not data["success_at_depth"]:
                continue
            
            # Find optimal depth (best success rate at lowest latency)
            success_rates = data["success_at_depth"]
            latencies = data["latency_at_depth"]
            
            # Score each depth: success_rate * (1 / latency)
            scores = {}
            for depth in success_rates.keys():
                latency = latencies.get(depth, 1)
                scores[depth] = (success_rates[depth] * 100) / (latency / 100)
            
            optimal_depth = max(scores.keys(), key=lambda d: scores[d])
            
            # Recommendation based on pattern
            optimal_success = success_rates[optimal_depth]
            if optimal_success > 0.95:
                recommendation = "allow_autonomy"
            elif optimal_depth >= 3:
                recommendation = "orchestrate"
            else:
                recommendation = "shallow_cascade"
            
            result[workflow] = {
                "optimal_depth": optimal_depth,
                "optimal_success_rate": optimal_success,
                "success_at_depth": success_rates,
                "latency_at_depth": latencies,
                "frequency_at_depth": data["frequency_at_depth"],
                "recommendation": recommendation
            }
        
        return result
    
    def generate_orchestrator_training_with_depth_insights(self) -> Dict:
        """
        Generate orchestrator training config enhanced with depth insights.
        
        Includes:
        - Workflows that work best with autonomy vs orchestration
        - Recommended depth limits per workflow
        - Performance baselines from each depth phase
        """
        
        workflow_insights = self.workflow_optimal_depth()
        phase_comparison = self.compare_phases()
        
        return {
            "metadata": {
                "analysis_type": "adaptive_depth_discovery",
                "phases_analyzed": len(self.phase_analyses),
                "total_calls": len(self.logs),
                "generated_at": datetime.utcnow().isoformat()
            },
            
            "phase_analyses": [
                analysis.to_dict() 
                for analysis in self.phase_analyses.values()
            ],
            
            "phase_comparison": phase_comparison,
            
            "workflow_optimal_depth_analysis": workflow_insights,
            
            "orchestrator_recommendations": {
                "workflows_that_need_orchestration": [
                    wf for wf, insight in workflow_insights.items()
                    if insight['recommendation'] == "orchestrate"
                ],
                "workflows_that_support_autonomy": [
                    wf for wf, insight in workflow_insights.items()
                    if insight['recommendation'] == "allow_autonomy"
                ],
                "recommended_default_depth": self._determine_optimal_default_depth(
                    workflow_insights
                ),
                "depth_variance_strategy": {
                    "description": "Different workflows can use different depth limits",
                    "per_workflow_config": self._generate_per_workflow_config(
                        workflow_insights
                    )
                }
            }
        }
    
    def _determine_optimal_default_depth(self, workflow_insights: Dict) -> int:
        """Determine optimal default depth for orchestrator"""
        
        # Default should support most workflows without hitting limits
        depths_needed = [
            insight['optimal_depth'] 
            for insight in workflow_insights.values()
        ]
        
        if not depths_needed:
            return 2
        
        # Use 75th percentile to balance autonomy and reliability
        sorted_depths = sorted(depths_needed)
        idx = int(len(sorted_depths) * 0.75)
        return sorted_depths[min(idx, len(sorted_depths)-1)]
    
    def _generate_per_workflow_config(self, workflow_insights: Dict) -> Dict:
        """Generate per-workflow depth configuration"""
        
        config = {}
        for workflow, insight in workflow_insights.items():
            config[workflow] = {
                "max_depth": insight['optimal_depth'],
                "reason": insight['recommendation'],
                "success_rate_target": insight['optimal_success_rate']
            }
        
        return config


# ============================================================================
# PART 3: EXPERIMENTAL DESIGN HELPER
# ============================================================================

class AdaptiveDepthExperimentDesigner:
    """Design and manage the adaptive depth discovery phases"""
    
    @staticmethod
    def create_experiment_schedule(start_date: datetime) -> List[DepthExperimentConfig]:
        """Create the 7-phase adaptive depth experiment schedule"""
        
        return [
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_1_DEPTH_1.value,
                max_depth=1,
                start_date=start_date,
                end_date=start_date + timedelta(days=7),
                duration_days=7,
                description="Baseline: Agents work independently, no inter-agent calls",
                hypothesis="Establish baseline behavior with zero cascading",
                expected_outcomes=[
                    "Single-agent workflows only",
                    "No depth limit violations",
                    "Baseline latency and success rate"
                ]
            ),
            
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_2_DEPTH_2.value,
                max_depth=2,
                start_date=start_date + timedelta(days=8),
                end_date=start_date + timedelta(days=21),
                duration_days=14,
                description="First cascading: Agents can call other agents once",
                hypothesis="Single-level agent calling enables most workflows",
                expected_outcomes=[
                    "Agent-to-agent calls enabled",
                    "Teams discover natural calling patterns",
                    "Some workflows still blocked"
                ]
            ),
            
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_3_DEPTH_3.value,
                max_depth=3,
                start_date=start_date + timedelta(days=22),
                end_date=start_date + timedelta(days=35),
                duration_days=14,
                description="Two-level cascading: Agents can call agents who call agents",
                hypothesis="Two-level cascading unblocks most complex workflows",
                expected_outcomes=[
                    "Complex multi-agent workflows enabled",
                    "Identify diminishing returns",
                    "Latency increases",
                    "Some error patterns emerge"
                ]
            ),
            
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_4_DEPTH_2_VALIDATION.value,
                max_depth=2,
                start_date=start_date + timedelta(days=36),
                end_date=start_date + timedelta(days=49),
                duration_days=14,
                description="Validation: Return to depth=2 to confirm patterns",
                hypothesis="Depth=2 performance should be consistent across time",
                expected_outcomes=[
                    "Consistent with Phase 2 metrics",
                    "Validate that workflows don't change over time",
                    "Identify any system evolution"
                ]
            ),
            
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_5_DEPTH_4.value,
                max_depth=4,
                start_date=start_date + timedelta(days=50),
                end_date=start_date + timedelta(days=63),
                duration_days=14,
                description="Outer bounds: Three-level cascading",
                hypothesis="Depth=4 shows clear diminishing returns",
                expected_outcomes=[
                    "Few workflows benefit from 4-level cascading",
                    "Significant latency increase",
                    "Higher error rates",
                    "Cost/benefit analysis unfavorable"
                ]
            ),
            
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_6_DEPTH_2_FINAL.value,
                max_depth=2,
                start_date=start_date + timedelta(days=64),
                end_date=start_date + timedelta(days=75),
                duration_days=12,
                description="Final validation: Confirm optimal depth",
                hypothesis="Depth=2 is optimal for production",
                expected_outcomes=[
                    "Metrics stable and consistent",
                    "All critical workflows enabled",
                    "Clear orchestration requirements identified"
                ]
            ),
            
            DepthExperimentConfig(
                phase_id=DepthPhase.PHASE_7_ADAPTIVE.value,
                max_depth=2,  # Default, but varies by workflow
                start_date=start_date + timedelta(days=76),
                end_date=start_date + timedelta(days=90),
                duration_days=15,
                description="Adaptive: Depth varies by workflow type",
                hypothesis="Different workflows can use different depth limits safely",
                expected_outcomes=[
                    "Complex workflows get depth=3+",
                    "Simple workflows get depth=1-2",
                    "Overall system performance improves",
                    "Orchestrator configuration is optimized"
                ]
            )
        ]


# ============================================================================
# PART 4: EXAMPLE USAGE AND REPORTING
# ============================================================================

def generate_synthetic_adaptive_logs() -> List[Dict]:
    """
    Generate synthetic A2A logs simulating 7 phases with varying depth limits.
    
    Simulates realistic patterns where:
    - Low depths (1-2) enable most workflows but block some cascades
    - Medium depths (2-3) enable most workflows with acceptable latency
    - High depths (4+) enable everything but with latency penalties
    """
    
    logs = []
    phases = [
        (DepthPhase.PHASE_1_DEPTH_1.value, 1, 0, 7),
        (DepthPhase.PHASE_2_DEPTH_2.value, 2, 7, 21),
        (DepthPhase.PHASE_3_DEPTH_3.value, 3, 21, 35),
        (DepthPhase.PHASE_4_DEPTH_2_VALIDATION.value, 2, 35, 49),
        (DepthPhase.PHASE_5_DEPTH_4.value, 4, 49, 63),
        (DepthPhase.PHASE_6_DEPTH_2_FINAL.value, 2, 63, 75),
        (DepthPhase.PHASE_7_ADAPTIVE.value, 2, 75, 90)
    ]
    
    agents = [
        "fundraising-agent",
        "business-development-agent",
        "field-operations-agent"
    ]
    
    call_count = 0
    for phase_id, max_depth, start_day, end_day in phases:
        calls_per_day = 40  # Consistent call volume
        total_phase_calls = calls_per_day * (end_day - start_day)
        
        for i in range(total_phase_calls):
            call_count += 1
            day = start_day + (i // calls_per_day)
            
            caller = agents[i % len(agents)]
            target = agents[(i + 1) % len(agents)]
            depth = (i // 100) % max_depth
            
            # Determine status based on depth and phase
            # Simulate: more cascades needed in later phases, get blocked at lower depths
            block_threshold = (max_depth * 15) + (100 - max_depth * 20)
            rand_val = (call_count * 23) % 100
            
            if depth >= max_depth:
                status = "depth_limit_exceeded"
                latency = 10
            elif rand_val > block_threshold:
                status = "timeout"
                latency = 5000
            else:
                status = "success"
                # Latency increases with depth
                if target == "business-development-agent":
                    latency = 100 + (depth * 50) + (i % 100)
                elif target == "fundraising-agent":
                    latency = 40 + (depth * 30) + (i % 60)
                else:
                    latency = 70 + (depth * 40) + (i % 80)
            
            logs.append({
                "timestamp": (
                    datetime.utcnow() - timedelta(days=90-day)
                ).isoformat(),
                "phase_id": phase_id,
                "max_depth": max_depth,
                "trace_id": f"trace-{call_count // 10}",
                "caller": caller,
                "target": target,
                "goal": f"goal_{i % 20}",
                "depth": depth,
                "status": status,
                "latency_ms": latency,
                "error": {"code": status} if status != "success" else None
            })
    
    return logs


def main():
    """Run adaptive depth discovery analysis"""
    
    print("=" * 90)
    print("ADAPTIVE DEPTH LIMITING DISCOVERY FRAMEWORK")
    print("=" * 90)
    print()


In [ ]:
# Create experiment schedule
    print("Experiment Schedule:")
    print("-" * 90)
    designer = AdaptiveDepthExperimentDesigner()
    schedule = designer.create_experiment_schedule(
        datetime.utcnow() - timedelta(days=90)
    )
    
    for config in schedule:
        print(f"{config.phase_id}:")
        print(f"  Duration: {config.duration_days} days | Max Depth: {config.max_depth}")
        print(f"  Hypothesis: {config.hypothesis}")
        print()


In [ ]:
# Generate synthetic logs
    print("Generating synthetic discovery logs (all 7 phases)...")
    logs = generate_synthetic_adaptive_logs()
    print(f"Generated {len(logs)} call logs across 7 phases")
    print()


In [ ]:
# Analyze
    print("Analyzing adaptive depth discovery data...")
    analyzer = AdaptiveDepthAnalyzer(logs)
    
    # Phase comparison
    print("\n" + "=" * 90)
    print("PHASE COMPARISON")
    print("=" * 90)


In [ ]:
comparison = analyzer.compare_phases()
    for phase_data in comparison["phases_ordered_by_depth"]:
        print(f"\nDepth {phase_data['depth']}:")
        print(f"  Success Rate: {phase_data['success_rate']*100:.1f}%")
        print(f"  Avg Latency: {phase_data['avg_latency_ms']:.0f}ms")
        print(f"  Enabled Workflows: {phase_data['enabled_workflows']}")
        print(f"  Blocked: {phase_data['blocked_workflows']}")


In [ ]:
print("\nOptimal Depth by Metric:")
    print(f"  For Success Rate: depth={comparison['optimal_depth_by_metric'].get('success_rate')}")
    print(f"  For Latency: depth={comparison['optimal_depth_by_metric'].get('latency')}")


In [ ]:
if comparison["diminishing_returns"]:
        print("\nDiminishing Returns Detected:")
        for dr in comparison["diminishing_returns"]:
            print(f"  {dr['between_depth']}: {dr['note']}")


In [ ]:
# Workflow analysis
    print("\n" + "=" * 90)
    print("WORKFLOW OPTIMAL DEPTH ANALYSIS")
    print("=" * 90)


In [ ]:
workflow_analysis = analyzer.workflow_optimal_depth()
    for workflow, insight in list(workflow_analysis.items())[:5]:
        print(f"\n{workflow}:")
        print(f"  Optimal Depth: {insight['optimal_depth']}")
        print(f"  Success Rate: {insight['optimal_success_rate']*100:.1f}%")
        print(f"  Recommendation: {insight['recommendation']}")
        print(f"  Success by depth: {insight['success_at_depth']}")


In [ ]:
# Generate training config
    print("\n" + "=" * 90)
    print("GENERATING ORCHESTRATOR TRAINING CONFIG")
    print("=" * 90)


In [ ]:
training_config = analyzer.generate_orchestrator_training_with_depth_insights()
    
    print("\nOrchestrator Recommendations:")
    recs = training_config["orchestrator_recommendations"]
    print(f"  Default Depth: {recs['recommended_default_depth']}")
    print(f"  Workflows needing orchestration: {len(recs['workflows_that_need_orchestration'])}")
    print(f"  Workflows supporting autonomy: {len(recs['workflows_that_support_autonomy'])}")


In [ ]:
# Save to file
    output_file = "/mnt/user-data/outputs/adaptive_depth_training_config.json"
    print(f"\nSaving training configuration to {output_file}...")
    with open(output_file, "w") as f:
        json.dump(training_config, f, indent=2, default=str)
    
    print("✓ Adaptive depth training configuration saved")
    print("\nKey Findings:")
    print("1. System behavior changes significantly across depth phases")
    print("2. Diminishing returns visible beyond depth=3")
    print("3. Different workflows have optimal depths")
    print("4. Production orchestrator can use adaptive depth per workflow")
    print("5. Training data enables data-driven orchestrator configuration")


In [ ]:
if __name__ == "__main__":
    main()
